# Modules Import


### In this section, we import all necessary libraries for our data processing and analysis tasks. This includes libraries for handling data structures (pandas, numpy), visualization (matplotlib, seaborn), working with JSON and compressed files (json, gzip), and text processing (nltk, re)

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import gzip
import ast
import re
import nltk
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from fastapi import FastAPI



# Stop words download
#nltk.download('vader_lexicon')
#nltk.download('stopwords')
#nltk.download('punkt')



# Datasets Import


## Games Dataset

### Here, we load and preprocess the dataset containing information about various games. This involves reading from a compressed .gz file, decoding JSON objects, and cleaning the data by dropping rows that are completely null.

In [2]:
# File Path
file_path = r"C:\Users\facue\OneDrive\Soy Henry\PI MLOps - STEAM\steam_games.json.gz"

# JSON Storage List
df_ga = []

# Reading gzip file
with gzip.open(file_path, 'r') as gz_file:
    for line in gz_file:
        # Decoding each line and conveting to JSON object
        json_obj = json.loads(line.decode('utf-8'))

        # Add JSON Object to list
        df_ga.append(json_obj)

# Convert JSON List to Dataframe
df_ga = pd.DataFrame(df_ga)

df_ga = df_ga.dropna(how = 'all')

df_ga.head()


,publisher,genres,app_name,title,url,release_date,tags,reviews_url,specs,price,early_access,id,developer
88310,Kotoshiro,"[Action, Casual, Indie, Simulation, Strategy]",Lost Summoner Kitty,Lost Summoner Kitty,http://store.steampowered.com/app/761140/Lost_...,2018-01-04,"[Strategy, Action, Indie, Casual, Simulation]",http://steamcommunity.com/app/761140/reviews/?...,[Single-player],4.99,False,761140,Kotoshiro
88311,"Making Fun, Inc.","[Free to Play, Indie, RPG, Strategy]",Ironbound,Ironbound,http://store.steampowered.com/app/643980/Ironb...,2018-01-04,"[Free to Play, Strategy, Indie, RPG, Card Game...",http://steamcommunity.com/app/643980/reviews/?...,"[Single-player, Multi-player, Online Multi-Pla...",Free To Play,False,643980,Secret Level SRL
88312,Poolians.com,"[Casual, Free to Play, Indie, Simulation, Sports]",Real Pool 3D - Poolians,Real Pool 3D - Poolians,http://store.steampowered.com/app/670290/Real_...,2017-07-24,"[Free to Play, Simulation, Sports, Casual, Ind...",http://steamcommunity.com/app/670290/reviews/?...,"[Single-player, Multi-player, Online Multi-Pla...",Free to Play,False,670290,Poolians.com
88313,彼岸领域,"[Action, Adventure, Casual]",弹炸人2222,弹炸人2222,http://store.steampowered.com/app/767400/2222/,2017-12-07,"[Action, Adventure, Casual]",http://steamcommunity.com/app/767400/reviews/?...,[Single-player],0.99,False,767400,彼岸领域
88314,NaN,NaN,Log Challenge,NaN,http://store.steampowered.com/app/773570/Log_C...,NaN,"[Action, Indie, Casual, Sports]",http://steamcommunity.com/app/773570/reviews/?...,"[Single-player, Full controller support, HTC V...",2.99,False,773570,NaN


In [3]:
# Deleting unnecesary columns
df_ga = df_ga.drop(columns=['publisher', 'title', 'url', 'early_access','reviews_url', 'tags'])


In [4]:
# Usaremos 'apply' para aplicar una función lambda a cada elemento de la columna 'price'.
df_ga['price'] = df_ga['price'].apply(lambda x: 0 if isinstance(x, str) and x not in [np.nan, None, 'NaN'] else x)

# Ahora, convierte toda la columna a un tipo numérico, reemplazando cualquier valor no convertible con NaN.
df_ga['price'] = pd.to_numeric(df_ga['price'], errors='coerce')


In [5]:
df_ga['release_date'].fillna(value='Unknown', inplace=True)

# Reemplazar valores no deseados con 'Unknown'
df_ga['release_date'].replace({'Soon..': 'Unknown'}, inplace=True)

# Convertir 'release_date' a datetime donde sea posible
df_ga['release_date'] = pd.to_datetime(df_ga['release_date'], errors='coerce')

# Filtrar las fechas que se convirtieron correctamente y extraer el año
df_ga['release_date'] = df_ga['release_date'].dt.year

# Para los casos donde la conversión falló y se convirtieron en NaT (Not a Time), establecer esos años como None o algún valor específico
df_ga['release_date'] = df_ga['release_date'].fillna('Unknown').astype('str')


In [6]:
df_ga.head()

,genres,app_name,release_date,specs,price,id,developer
88310,"[Action, Casual, Indie, Simulation, Strategy]",Lost Summoner Kitty,2018.0,[Single-player],4.99,761140,Kotoshiro
88311,"[Free to Play, Indie, RPG, Strategy]",Ironbound,2018.0,"[Single-player, Multi-player, Online Multi-Pla...",0.00,643980,Secret Level SRL
88312,"[Casual, Free to Play, Indie, Simulation, Sports]",Real Pool 3D - Poolians,2017.0,"[Single-player, Multi-player, Online Multi-Pla...",0.00,670290,Poolians.com
88313,"[Action, Adventure, Casual]",弹炸人2222,2017.0,[Single-player],0.99,767400,彼岸领域
88314,NaN,Log Challenge,Unknown,"[Single-player, Full controller support, HTC V...",2.99,773570,NaN


In [7]:
# Converting all non-list values in empty lists
df_ga['genres'] = df_ga['genres'].apply(lambda x: x if isinstance(x, list) else [])

# Genres subset
genres_set = set(genre for sublist in df_ga['genres'] for genre in sublist)

# Creating a Column for each genre
for genre in genres_set:
    df_ga[f'genre_{genre}'] = df_ga['genres'].apply(lambda x: 1 if genre in x else 0)


# Droping original column
df_ga.drop('genres', axis=1, inplace=True)

df_ga.head()

,app_name,release_date,specs,price,id,developer,genre_Audio Production,genre_Sports,genre_Photo Editing,genre_Strategy,...,genre_Design &amp; Illustration,genre_Utilities,genre_Early Access,genre_Indie,genre_Racing,genre_RPG,genre_Adventure,genre_Simulation,genre_Software Training,genre_Video Production
88310,Lost Summoner Kitty,2018.0,[Single-player],4.99,761140,Kotoshiro,0,0,0,1,...,0,0,0,1,0,0,0,1,0,0
88311,Ironbound,2018.0,"[Single-player, Multi-player, Online Multi-Pla...",0.00,643980,Secret Level SRL,0,0,0,1,...,0,0,0,1,0,1,0,0,0,0
88312,Real Pool 3D - Poolians,2017.0,"[Single-player, Multi-player, Online Multi-Pla...",0.00,670290,Poolians.com,0,1,0,0,...,0,0,0,1,0,0,0,1,0,0
88313,弹炸人2222,2017.0,[Single-player],0.99,767400,彼岸领域,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
88314,Log Challenge,Unknown,"[Single-player, Full controller support, HTC V...",2.99,773570,NaN,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# Converting all non-list values in empty lists
df_ga['specs'] = df_ga['specs'].apply(lambda x: x if isinstance(x, list) else [])

# Spec subset
specs_set = set(spec for sublist in df_ga['specs'] for spec in sublist)

# Creating a Column for each specs
for spec in specs_set:
    df_ga[f'spec_{spec}'] = df_ga['specs'].apply(lambda x: 1 if spec in x else 0)


# Droping original column
df_ga.drop('specs', axis=1, inplace=True)

df_ga.head()

,app_name,release_date,price,id,developer,genre_Audio Production,genre_Sports,genre_Photo Editing,genre_Strategy,genre_Education,...,spec_SteamVR Collectibles,spec_Full controller support,spec_Single-player,spec_Game demo,spec_Steam Achievements,spec_Steam Leaderboards,spec_Local Multi-Player,spec_Multi-player,spec_Keyboard / Mouse,spec_Stats
88310,Lost Summoner Kitty,2018.0,4.99,761140,Kotoshiro,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
88311,Ironbound,2018.0,0.00,643980,Secret Level SRL,0,0,0,1,0,...,0,0,1,0,1,0,0,1,0,0
88312,Real Pool 3D - Poolians,2017.0,0.00,670290,Poolians.com,0,1,0,0,0,...,0,0,1,0,0,0,0,1,0,1
88313,弹炸人2222,2017.0,0.99,767400,彼岸领域,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
88314,Log Challenge,Unknown,2.99,773570,NaN,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0


In [9]:
df_ga

,app_name,release_date,price,id,developer,genre_Audio Production,genre_Sports,genre_Photo Editing,genre_Strategy,genre_Education,...,spec_SteamVR Collectibles,spec_Full controller support,spec_Single-player,spec_Game demo,spec_Steam Achievements,spec_Steam Leaderboards,spec_Local Multi-Player,spec_Multi-player,spec_Keyboard / Mouse,spec_Stats
88310,Lost Summoner Kitty,2018.0,4.99,761140,Kotoshiro,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
88311,Ironbound,2018.0,0.00,643980,Secret Level SRL,0,0,0,1,0,...,0,0,1,0,1,0,0,1,0,0
88312,Real Pool 3D - Poolians,2017.0,0.00,670290,Poolians.com,0,1,0,0,0,...,0,0,1,0,0,0,0,1,0,1
88313,弹炸人2222,2017.0,0.99,767400,彼岸领域,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
88314,Log Challenge,Unknown,2.99,773570,NaN,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120440,Colony On Mars,2018.0,1.99,773640,"Nikita ""Ghost_RUS""",0,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0
120441,LOGistICAL: South Africa,2018.0,4.99,733530,Sacada,0,0,0,1,0,...,0,0,1,0,1,1,0,0,0,1
120442,Russian Roads,2018.0,1.99,610660,Laush Dmitriy Sergeevich,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
120443,EXIT 2 - Directions,2017.0,4.99,658870,"xropi,stev3ns",0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0


In [10]:
df_ga['release_date'].isna().sum()

0

In [11]:
game_info = df_ga[df_ga['id'] == '10']
game_info

,app_name,release_date,price,id,developer,genre_Audio Production,genre_Sports,genre_Photo Editing,genre_Strategy,genre_Education,...,spec_SteamVR Collectibles,spec_Full controller support,spec_Single-player,spec_Game demo,spec_Steam Achievements,spec_Steam Leaderboards,spec_Local Multi-Player,spec_Multi-player,spec_Keyboard / Mouse,spec_Stats
120416,Counter-Strike,2000.0,9.99,10,Valve,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [12]:
df_ga.columns

Index(['app_name', 'release_date', 'price', 'id', 'developer',
       'genre_Audio Production', 'genre_Sports', 'genre_Photo Editing',
       'genre_Strategy', 'genre_Education', 'genre_Casual', 'genre_Action',
       'genre_Animation &amp; Modeling', 'genre_Accounting',
       'genre_Web Publishing', 'genre_Free to Play',
       'genre_Massively Multiplayer', 'genre_Design &amp; Illustration',
       'genre_Utilities', 'genre_Early Access', 'genre_Indie', 'genre_Racing',
       'genre_RPG', 'genre_Adventure', 'genre_Simulation',
       'genre_Software Training', 'genre_Video Production',
       'spec_Cross-Platform Multiplayer', 'spec_Includes Source SDK',
       'spec_Steam Workshop', 'spec_Gamepad', 'spec_Mods (require HL2)',
       'spec_Downloadable Content', 'spec_Mods', 'spec_Room-Scale',
       'spec_Online Multi-Player', 'spec_Steam Trading Cards',
       'spec_Local Co-op', 'spec_Seated', 'spec_Commentary available',
       'spec_Steam Cloud', 'spec_MMO', 'spec_Includes level

## Reviews Dataset

### Similarly, for the reviews dataset, we process each line from a JSON file, handling potential errors and converting lines into a list of dictionaries, which we then transform into a DataFrame. This step is crucial for structuring the reviews data for further analysis.

In [13]:
data_list = []


# File Path
file_path = r"C:\Users\facue\OneDrive\Soy Henry\PI MLOps - STEAM\user_reviews.json\australian_user_reviews.json"


# Processing each line
with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            # Usar ast.literal_eval para convertir la línea en un diccionario
            json_data = ast.literal_eval(line)
            data_list.append(json_data)
        except ValueError as e:
            print(f"Error in line: {line}")
            continue

# Crafting DataFrame from dic list
df_re = pd.DataFrame(data_list)

# Removing user_url attribute
df_re = df_re.drop(['user_url'], axis= 1)

# Removing Null Registers
df_re = df_re.dropna(how = 'all')

# Reset index
df_re = df_re.reset_index(drop=True)

In [14]:
df_re

,user_id,reviews
0,76561197970982479,"[{'funny': '', 'posted': 'Posted November 5, 2..."
1,js41637,"[{'funny': '', 'posted': 'Posted June 24, 2014..."
2,evcentric,"[{'funny': '', 'posted': 'Posted February 3.',..."
3,doctr,"[{'funny': '', 'posted': 'Posted October 14, 2..."
4,maplemage,"[{'funny': '3 people found this review funny',..."
...,...,...
25794,76561198306599751,"[{'funny': '', 'posted': 'Posted May 31.', 'la..."
25795,Ghoustik,"[{'funny': '', 'posted': 'Posted June 17.', 'l..."
25796,76561198310819422,"[{'funny': '1 person found this review funny',..."
25797,76561198312638244,"[{'funny': '', 'posted': 'Posted July 21.', 'l..."


In [15]:
# Exploding reviews column
df_re= df_re.explode('reviews')

# Filtering registers with one relevant key
df_re = df_re[df_re['reviews'].apply(lambda x: isinstance(x, dict) and 'review' in x)]

# Attributes assign
df_re['funny'] = df_re['reviews'].apply(lambda x: x.get('funny'))
df_re['posted'] = df_re['reviews'].apply(lambda x: x.get('posted'))
df_re['last_edited'] = df_re['reviews'].apply(lambda x: x.get('last_edited'))
df_re['item_id'] = df_re['reviews'].apply(lambda x: x.get('item_id'))
df_re['helpful'] = df_re['reviews'].apply(lambda x: x.get('helpful'))
df_re['recommend'] = df_re['reviews'].apply(lambda x: x.get('recommend'))
df_re['review_text'] = df_re['reviews'].apply(lambda x: x.get('review'))

# Removing 'Reviews' column
df_re.drop(columns=['reviews'], inplace=True)

# Removing Duplicates
df_re.drop_duplicates(inplace=True)



In [16]:
df_re = df_re.drop(columns = ['funny', 'last_edited','helpful'])

In [17]:
df_re

,user_id,posted,item_id,recommend,review_text
0,76561197970982479,"Posted November 5, 2011.",1250,True,Simple yet with great replayability. In my opi...
0,76561197970982479,"Posted July 15, 2011.",22200,True,It's unique and worth a playthrough.
0,76561197970982479,"Posted April 21, 2011.",43110,True,Great atmosphere. The gunplay can be a bit chu...
1,js41637,"Posted June 24, 2014.",251610,True,I know what you think when you see this title ...
1,js41637,"Posted September 8, 2013.",227300,True,For a simple (it's actually not all that simpl...
...,...,...,...,...,...
25797,76561198312638244,Posted July 10.,70,True,a must have classic from steam definitely wort...
25797,76561198312638244,Posted July 8.,362890,True,this game is a perfect remake of the original ...
25798,LydiaMorley,Posted July 3.,273110,True,had so much fun plaing this and collecting res...
25798,LydiaMorley,Posted July 20.,730,True,:D


### NLP Classification

#### In order to analyze the sentiment of user reviews, we first preprocess the text to remove URLs, special characters, and numbers, and to tokenize the text. This preprocessing step is vital for reducing noise in the text data and improving the performance of our sentiment analysis.

In [18]:
# Start Stemmer
stemmer = PorterStemmer()

def preprocess_text(text):
    '''
    Function to preprocess text by lowering case, removing URLs, special characters, and numbers,
    tokenizing, removing stopwords, and applying stemming.
    '''
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'http\S+', '', text)  # URLs delete
    text = re.sub(r'[^a-z\s]', '', text)  # Numbers and Special Characters delete
    tokens = word_tokenize(text)  # Tokenization
    tokens = [stemmer.stem(word) for word in tokens if word not in set(stopwords.words('english'))]  # Stopwords delete and Steeming
    return ' '.join(tokens)

# Deploying preprocessing function
df_re['cleaned_review'] = df_re['review_text'].apply(preprocess_text) 



#### We utilize the VADER tool from the nltk library to perform sentiment analysis on the preprocessed review texts. This tool is particularly suited for texts from social media and similar contexts due to its sensitivity to both the polarity and intensity of emotions.

In [19]:
# Creating Analyzer Instance
sid = SentimentIntensityAnalyzer()

def sentiment_analysis_vader(text):
    if pd.isna(text):
        return 1  # Neutral if there is no text
    
    # Sentiment Score
    scores = sid.polarity_scores(text)

    # Score Assign
    compound = scores['compound']
    if compound >= 0.05:
        return 2  # Positive
    elif compound <= -0.05:
        return 0  # Negative
    else:
        return 1  # Neutral

# Deploying function to Dataset
df_re['sentiment_analysis'] = df_re['cleaned_review'].apply(sentiment_analysis_vader)



In [20]:
df_re

,user_id,posted,item_id,recommend,review_text,cleaned_review,sentiment_analysis
0,76561197970982479,"Posted November 5, 2011.",1250,True,Simple yet with great replayability. In my opi...,simpl yet great replay opinion zombi hord team...,2
0,76561197970982479,"Posted July 15, 2011.",22200,True,It's unique and worth a playthrough.,uniqu worth playthrough,2
0,76561197970982479,"Posted April 21, 2011.",43110,True,Great atmosphere. The gunplay can be a bit chu...,great atmospher gunplay bit chunki time end da...,2
1,js41637,"Posted June 24, 2014.",251610,True,I know what you think when you see this title ...,know think see titl barbi dreamhous parti inti...,0
1,js41637,"Posted September 8, 2013.",227300,True,For a simple (it's actually not all that simpl...,simpl actual simpl truck drive simul quit fun ...,2
...,...,...,...,...,...,...,...
25797,76561198312638244,Posted July 10.,70,True,a must have classic from steam definitely wort...,must classic steam definit worth buy,2
25797,76561198312638244,Posted July 8.,362890,True,this game is a perfect remake of the original ...,game perfect remak origin half life person one...,2
25798,LydiaMorley,Posted July 3.,273110,True,had so much fun plaing this and collecting res...,much fun pla collect resourc xd first tri kill...,2
25798,LydiaMorley,Posted July 20.,730,True,:D,,1


In [21]:
review_info = df_re[df_re['sentiment_analysis'] == 0]
review_info


,user_id,posted,item_id,recommend,review_text,cleaned_review,sentiment_analysis
1,js41637,"Posted June 24, 2014.",251610,True,I know what you think when you see this title ...,know think see titl barbi dreamhous parti inti...,0
2,evcentric,"Posted October 15, 2014.",263360,True,"Random drops and random quests, with stat poin...",random drop random quest stat point anim style...,0
3,doctr,"Posted November 22, 2012.",207610,True,The ending to this game is.... ♥♥♥♥♥♥♥.... Jus...,end game buy youll invest im automat preorder ...,0
5,Wackky,"Posted December 24, 2012.",207610,True,"It reminds me of that TV Show called ""The Walk...",remind tv show call walk dead,0
8,76561198089393905,"Posted February 1, 2015.",72850,True,"Killed the Emperor, nobody cared and got away ...",kill emperor nobodi care got away accident kil...,0
...,...,...,...,...,...,...,...
25753,112367263762,Posted July 31.,730,True,Can't count how many times i rage quitted onl...,cant count mani time rage quit come back,0
25765,76561198251004808,"Posted October 10, 2015.",253980,True,Awesome fantasy game if you don't mind the gra...,awesom fantasi game dont mind graphic game ill...,0
25769,72947282842,"Posted October 31, 2015.",730,True,Prettyy Mad Game,prettyy mad game,0
25772,iwishihadaids,Posted February 25.,391460,False,Bad,bad,0


In [22]:
df_re['sentiment_analysis'].value_counts()


2    34274
1    15350
0     8807
Name: sentiment_analysis, dtype: int64

### Items Dateset

#### In this section, we focus on the dataset that contains information about users' items. This includes the games or software owned by users on the Steam platform. We start by loading the data from a JSON file and proceed to clean it by removing entries that don't provide meaningful information.

In [23]:
data_list = []


# File Path
file_path = r"C:\Users\facue\OneDrive\Soy Henry\PI MLOps - STEAM\users_items.json\australian_users_items.json"


#Open file processing all lines
with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            # Converting to Dict
            json_data = ast.literal_eval(line)
            data_list.append(json_data)
        except ValueError as e:
            print(f"Error in line: {line}")
            continue

#Converting Dict list to Dataframe
df_it = pd.DataFrame(data_list)

df_it = df_it.drop(['user_url'], axis= 1)

In [24]:
df_it

,user_id,items_count,steam_id,items
0,76561197970982479,277,76561197970982479,"[{'item_id': '10', 'item_name': 'Counter-Strik..."
1,js41637,888,76561198035864385,"[{'item_id': '10', 'item_name': 'Counter-Strik..."
2,evcentric,137,76561198007712555,"[{'item_id': '1200', 'item_name': 'Red Orchest..."
3,Riot-Punch,328,76561197963445855,"[{'item_id': '10', 'item_name': 'Counter-Strik..."
4,doctr,541,76561198002099482,"[{'item_id': '300', 'item_name': 'Day of Defea..."
...,...,...,...,...
88305,76561198323066619,22,76561198323066619,"[{'item_id': '413850', 'item_name': 'CS:GO Pla..."
88306,76561198326700687,177,76561198326700687,"[{'item_id': '11020', 'item_name': 'TrackMania..."
88307,XxLaughingJackClown77xX,0,76561198328759259,[]
88308,76561198329548331,7,76561198329548331,"[{'item_id': '304930', 'item_name': 'Unturned'..."


In [25]:
#Delete users with no items
df_it = df_it[df_it['items_count'] != 0]

In [26]:
# Convert item to dict
def convert_to_dict(item):
    try:
        return ast.literal_eval(item)
    except:
        return None

# Applying function to each item register
df_it['items'] = df_it['items'].apply(lambda x: convert_to_dict(x) if isinstance(x, str) else x)

# Apply explode to items as a dict
df_it = df_it.explode('items')

# Assign attributes to new columns
df_it['item_id'] = df_it['items'].apply(lambda x: x.get('item_id') if isinstance(x, dict) else None)
df_it['item_name'] = df_it['items'].apply(lambda x: x.get('item_name') if isinstance(x, dict) else None)
df_it['playtime_forever'] = df_it['items'].apply(lambda x: x.get('playtime_forever') if isinstance(x, dict) else None)
df_it['playtime_2weeks'] = df_it['items'].apply(lambda x: x.get('playtime_2weeks') if isinstance(x, dict) else None)

# Drop 'items' columns
df_it.drop(columns=['items'], inplace=True)




C:\Users\facue\AppData\Local\Temp/ipykernel_7392/2283612002.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_it['items'] = df_it['items'].apply(lambda x: convert_to_dict(x) if isinstance(x, str) else x)


In [27]:
df_it.drop_duplicates(inplace = True)

In [28]:
df_it

,user_id,items_count,steam_id,item_id,item_name,playtime_forever,playtime_2weeks
0,76561197970982479,277,76561197970982479,10,Counter-Strike,6,0
0,76561197970982479,277,76561197970982479,20,Team Fortress Classic,0,0
0,76561197970982479,277,76561197970982479,30,Day of Defeat,7,0
0,76561197970982479,277,76561197970982479,40,Deathmatch Classic,0,0
0,76561197970982479,277,76561197970982479,50,Half-Life: Opposing Force,0,0
...,...,...,...,...,...,...,...
88308,76561198329548331,7,76561198329548331,346330,BrainBread 2,0,0
88308,76561198329548331,7,76561198329548331,373330,All Is Dust,0,0
88308,76561198329548331,7,76561198329548331,388490,One Way To Die: Steam Edition,3,3
88308,76561198329548331,7,76561198329548331,521570,You Have 10 Seconds 2,4,4


In [29]:
# Guardar el DataFrame de juegos
df_ga.to_csv('../datasets/steam_games_cleaned.csv', index=False)

# Guardar el DataFrame de reseñas
df_re.to_csv('../datasets/steam_reviews_cleaned.csv', index=False)

# Guardar el DataFrame de items
df_it.to_csv('../datasets/steam_items_cleaned.csv', index=False)